# Kickstarter Funding Prediction — NLP Embedding Baselines
### 4 Embeddings (Word2Vec, SBERT, DistilBERT, BERT) x 3 Models (XGBoost, CatBoost, Random Forest) = 12 Baseline Runs

This notebook merges each of the 4 NLP embedding sets (generated in `kickstarter_nlp_embeddings.ipynb`)
onto the existing tabular `ML_train.csv` / `ML_test.csv` features, and trains **baseline (default
hyperparameter)** XGBoost, CatBoost, and Random Forest models on each combination.

Conventions match the existing `kickstarter_all_models_ensembles_final.ipynb` baseline notebook:
- Target: `log_target`
- Dropped from X: `id`, `target_usd`, `log_target`, `goal_usd`
- Same `evaluate_model()` metrics function (MAE/MSE/RMSE/R2 on log scale and USD scale, plus RMSLE)
- Same baseline hyperparameters for XGBoost, CatBoost, Random Forest

**Output:** `kickstarter_nlp_embedding_baseline_12_results.csv`


## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)

try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("xgboost is not installed. Run: pip install xgboost")

try:
    from catboost import CatBoostRegressor
except ImportError:
    raise ImportError("catboost is not installed. Run: pip install catboost")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42


## 2. Load Tabular Train/Test and Embedding Files

In [2]:
TRAIN_FILE = r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv"
TEST_FILE = r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv"

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16000, 81)
Test shape: (4000, 81)


In [3]:
EMBEDDING_PATH = r"E:\NSU\cse445\EDA attempt3\NLP models\embeddings"

EMBEDDING_FILES = {
    # "Word2Vec":   EMBEDDING_PATH + r"\word2vec_embeddings.csv",
    # "SBERT":      EMBEDDING_PATH + r"\sbert_embeddings.csv",
    # "DistilBERT": EMBEDDING_PATH + r"\distilbert_embeddings.csv",
    # "BERT":       EMBEDDING_PATH + r"\bert_embeddings.csv",
    "TFIDF":      EMBEDDING_PATH + r"\tfidf_embeddings.csv",
}

embeddings_raw = {}
for name, path in EMBEDDING_FILES.items():
    emb_df = pd.read_csv(path)
    embeddings_raw[name] = emb_df
    print(f"{name}: {emb_df.shape}")


TFIDF: (20000, 5001)


## 3. PCA Reduction (fit on train rows only)

Each embedding set is reduced to 20 components to avoid dominating the ~80 existing tabular
features. PCA is fit only on the rows that belong to `train_df["id"]`, then applied to both
train and test rows to prevent leakage.


In [4]:
N_COMPONENTS = 20

train_ids = set(train_df["id"])
test_ids = set(test_df["id"])

embeddings_reduced = {}

for name, emb_df in embeddings_raw.items():
    feature_cols = [c for c in emb_df.columns if c != "id"]

    train_mask = emb_df["id"].isin(train_ids)
    test_mask = emb_df["id"].isin(test_ids)

    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    pca.fit(emb_df.loc[train_mask, feature_cols])

    reduced_all = pca.transform(emb_df[feature_cols])

    reduced_cols = [f"{name.lower()}_pca_{i}" for i in range(N_COMPONENTS)]
    reduced_df = pd.DataFrame(reduced_all, columns=reduced_cols)
    reduced_df.insert(0, "id", emb_df["id"].values)

    embeddings_reduced[name] = reduced_df

    explained = pca.explained_variance_ratio_.sum()
    print(f"{name}: reduced to {N_COMPONENTS} dims, explained variance = {explained:.3f}")


TFIDF: reduced to 20 dims, explained variance = 0.162


## 4. Evaluation Helper (same as baseline notebook)

In [5]:
def evaluate_model(model_name, y_true_log, pred_log, actual_usd):
    pred_usd = np.expm1(pred_log)
    pred_usd = np.clip(pred_usd, a_min=0, a_max=None)

    mae_log = mean_absolute_error(y_true_log, pred_log)
    mse_log = mean_squared_error(y_true_log, pred_log)
    rmse_log = np.sqrt(mse_log)
    r2_log = r2_score(y_true_log, pred_log)

    mae_usd = mean_absolute_error(actual_usd, pred_usd)
    mse_usd = mean_squared_error(actual_usd, pred_usd)
    rmse_usd = np.sqrt(mse_usd)
    r2_usd = r2_score(actual_usd, pred_usd)

    rmsle = np.sqrt(mean_squared_log_error(actual_usd, pred_usd))

    metrics = {
        "Model":      model_name,
        "MAE_log":    mae_log,
        "MSE_log":    mse_log,
        "RMSE_log":   rmse_log,
        "R2_log":     r2_log,
        "MAE_USD":    mae_usd,
        "MSE_USD":    mse_usd,
        "RMSE_USD":   rmse_usd,
        "R2_USD":     r2_usd,
        "RMSLE":      rmsle
    }

    return metrics, pred_usd


## 5. Merge Function + Feature Setup

For each embedding, merge its reduced PCA columns onto `train_df` / `test_df` via `id`,
then build `X_train` / `X_test` / `y_train` / `y_test` using the same `DROP_FROM_X` convention
as the baseline notebook.


In [6]:
TARGET_RAW = "target_usd"
TARGET_LOG = "log_target"

DROP_FROM_X = [
    "id",
    "target_usd",
    "log_target",
    "goal_usd"
]

def build_features_for_embedding(embedding_name):
    reduced_df = embeddings_reduced[embedding_name]

    merged_train = train_df.merge(reduced_df, on="id", how="left")
    merged_test = test_df.merge(reduced_df, on="id", how="left")

    assert merged_train.shape[0] == train_df.shape[0], "Row count mismatch after merge (train)"
    assert merged_test.shape[0] == test_df.shape[0], "Row count mismatch after merge (test)"

    pca_cols = [c for c in reduced_df.columns if c != "id"]
    assert merged_train[pca_cols].isna().sum().sum() == 0, f"NaNs found after merging {embedding_name} (train)"
    assert merged_test[pca_cols].isna().sum().sum() == 0, f"NaNs found after merging {embedding_name} (test)"

    X_train = merged_train.drop(columns=DROP_FROM_X, errors="ignore")
    X_test = merged_test.drop(columns=DROP_FROM_X, errors="ignore")

    y_train = merged_train[TARGET_LOG].copy()
    y_test = merged_test[TARGET_LOG].copy()

    actual_usd = merged_test[TARGET_RAW].to_numpy()

    assert list(X_train.columns) == list(X_test.columns)

    return X_train, X_test, y_train, y_test, actual_usd


## 6. Baseline Model Definitions

Same default hyperparameters as the original baseline notebook (`kickstarter_all_models_ensembles_final.ipynb`).


In [7]:
def make_random_forest():
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

def make_xgboost():
    return XGBRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.0,
        objective="reg:squarederror",
        eval_metric="rmse",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

def make_catboost():
    return CatBoostRegressor(
        iterations=800,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

MODEL_FACTORIES = {
    "Random Forest": make_random_forest,
    "XGBoost":       make_xgboost,
    "CatBoost":      make_catboost,
}


## 7. Run All 15 Combinations (5 Embeddings x 3 Models)

For each embedding, build features once, then train/evaluate all 3 models.


In [8]:
all_results = []

for embedding_name in embeddings_reduced.keys():
    print(f"\n{'='*60}")
    print(f"Embedding: {embedding_name}")
    print(f"{'='*60}")

    X_train, X_test, y_train, y_test, actual_usd = build_features_for_embedding(embedding_name)
    print(f"Feature count: {X_train.shape[1]}")

    for model_name, factory in MODEL_FACTORIES.items():
        model = factory()

        start_time = time.time()

        if model_name == "XGBoost":
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        elif model_name == "CatBoost":
            model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=False)
        else:
            model.fit(X_train, y_train)

        training_time = time.time() - start_time

        pred_log = model.predict(X_test)

        run_label = f"{embedding_name} + {model_name}"
        metrics, pred_usd = evaluate_model(run_label, y_test, pred_log, actual_usd)
        metrics["Embedding"] = embedding_name
        metrics["Base_Model"] = model_name
        metrics["Training_Time_Seconds"] = training_time

        all_results.append(metrics)

        print(f"  {model_name}: RMSLE={metrics['RMSLE']:.4f}, R2_log={metrics['R2_log']:.4f}, "
              f"time={training_time:.1f}s")

print(f"\nTotal runs completed: {len(all_results)}")



Embedding: TFIDF
Feature count: 97
  Random Forest: RMSLE=2.3028, R2_log=0.4356, time=4.9s
  XGBoost: RMSLE=2.1181, R2_log=0.5224, time=5.5s
  CatBoost: RMSLE=2.1775, R2_log=0.4952, time=10.9s

Total runs completed: 3


## 8. Combine and Display Results

In [9]:
results_df = pd.DataFrame(all_results)

results_df = results_df[
    [
        "Embedding",
        "Base_Model",
        "Model",
        "MAE_log",
        "MSE_log",
        "RMSE_log",
        "R2_log",
        "MAE_USD",
        "MSE_USD",
        "RMSE_USD",
        "R2_USD",
        "RMSLE",
        "Training_Time_Seconds"
    ]
]

results_df = results_df.sort_values("RMSLE").reset_index(drop=True)
display(results_df)


,Embedding,Base_Model,Model,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,TFIDF,XGBoost,TFIDF + XGBoost,1.556848,4.487013,2.118257,0.522391,15923.452413,1.103115e+10,105029.288945,0.065275,2.118082,5.485248
1,TFIDF,CatBoost,TFIDF + CatBoost,1.612824,4.742204,2.177660,0.495228,16227.475634,1.129896e+10,106296.561184,0.042582,2.177547,10.891106
2,TFIDF,Random Forest,TFIDF + Random Forest,1.733656,5.302737,2.302767,0.435563,16801.587263,1.161765e+10,107785.223784,0.015578,2.302767,4.879919


In [10]:
best_row = results_df.iloc[0]
print("Best embedding + model combination by RMSLE:")
print(best_row["Model"])
print(f"RMSLE:   {best_row['RMSLE']:.4f}")
print(f"R2_log:  {best_row['R2_log']:.4f}")
print(f"MAE_USD: \${best_row['MAE_USD']:,.2f}")


Best embedding + model combination by RMSLE:
TFIDF + XGBoost
RMSLE:   2.1181
R2_log:  0.5224
MAE_USD: \$15,923.45


## 9. Save Results

In [11]:
results_df.to_csv("tfidf_result.csv", index=False)
print("Saved: tfidf_result.csv")


Saved: tfidf_result.csv


## Summary

This notebook produced **12 baseline runs** (4 embeddings x 3 models, all default hyperparameters):

| Embedding | Models |
|---|---|
| Word2Vec | Random Forest, XGBoost, CatBoost |
| SBERT | Random Forest, XGBoost, CatBoost |
| DistilBERT | Random Forest, XGBoost, CatBoost |
| BERT | Random Forest, XGBoost, CatBoost |

**Next step (not part of this notebook):** apply hyperparameter tuning (RandomizedSearchCV,
GridSearchCV, BayesSearchCV) to each of these 12 combinations, and compare tuned vs. baseline
results, alongside the original tabular-only baseline from `kickstarter_all_models_ensembles_final.ipynb`.
